# Chapter 2 — Matrices and Linear Transformations

Companion notebook: every figure and number in the chapter is produced here.

Run: `clae-py -m nbconvert --to notebook --execute --inplace ch02.ipynb`

## 2.1 — The matrix is a verb: a derivative is a matrix

In [1]:
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [2]:
n = 1000
x = np.linspace(0, 2*np.pi, n)
h = x[1] - x[0]
D = (np.eye(n, k=1) - np.eye(n)) / h   # forward difference

err = np.abs(D @ np.sin(x) - np.cos(x))[:-1].max()
print(f'max |D @ sin - cos|: {err:.4f}')

max |D @ sin - cos|: 0.0031


In [3]:
plt.figure(figsize=(8, 4.5))
plt.plot(x, np.sin(x), label='input: sin(x)', alpha=0.5)
plt.plot(x[:-1], (D @ np.sin(x))[:-1], label='output: D @ sin(x)',
         lw=2.5)
plt.plot(x, np.cos(x), '--', label='cos(x)', color='k', lw=1)
plt.legend(); plt.grid(True, alpha=0.3)
plt.title('The matrix D takes a derivative')
plt.savefig('figures/fig_matrix_derivative.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [4]:
K = np.eye(n, k=1) - 2*np.eye(n) + np.eye(n, k=-1)   # secondDiff
K = K / h**2
err2 = np.abs((K @ np.sin(x) + np.sin(x))[1:-1]).max()
print(f'max |K @ sin + sin|: {err2:.6f}   (K @ sin = -sin: '
      f'the second derivative)')

max |K @ sin + sin|: 0.000003   (K @ sin = -sin: the second derivative)


## 2.2 — One product, two views; composition; the undo

In [5]:
A = np.array([[1, 2], [3, 4], [5, 6]])
x = np.array([7, 8])

by_rows = np.array([row @ x for row in A])
by_cols = x[0] * A[:, 0] + x[1] * A[:, 1]
print('A @ x     :', A @ x)
print('row view  :', by_rows)
print('column view:', by_cols)

A @ x     : [23 53 83]
row view  : [23 53 83]
column view: [23 53 83]


In [6]:
def R(t):
    return np.array([[np.cos(t), -np.sin(t)],
                     [np.sin(t),  np.cos(t)]])

a, b = 0.7, 0.4
print('max |R(a) @ R(b) - R(a+b)|:',
      np.abs(R(a) @ R(b) - R(a + b)).max())

max |R(a) @ R(b) - R(a+b)|: 1.6653345369377348e-16


In [7]:
A3 = np.array([[1, 0, 0], [-1, 1, 0], [0, -1, 1]])  # difference
print('inverse of the difference matrix:')
print(np.linalg.inv(A3))                            # running sums
b = np.array([1, 3, 5])
print('x = inv(A) @ (1,3,5):', np.linalg.inv(A3) @ b)

inverse of the difference matrix:
[[1. 0. 0.]
 [1. 1. 0.]
 [1. 1. 1.]]
x = inv(A) @ (1,3,5): [1. 4. 9.]


## 2.3 — Geometric effects: rotate, scale, project

In [8]:
t = np.linspace(0, 2*np.pi, 100)
circle = np.vstack([np.cos(t), np.sin(t)])
u = np.array([2.0, 1.0])
P = np.outer(u, u) / (u @ u)               # projection onto u
S = np.diag([2.0, 0.5])                    # scaling
effects = [(R(np.pi/6), 'rotation R(30\u00b0)'),
           (S, 'scaling diag(2, 1/2)'),
           (P, 'projection onto u')]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (M, name) in zip(axes, effects):
    out = M @ circle
    ax.plot(circle[0], circle[1], alpha=0.35, label='input')
    ax.plot(out[0], out[1], lw=2.5, label='output')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(name); ax.legend(fontsize=8)
    ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6)
plt.savefig('figures/fig_geometric_effects.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [9]:
print('P @ P == P?  max diff:', np.abs(P @ P - P).max())
v = np.array([1.0, 2.0])
print('residual . u =', (v - P @ v) @ u)   # orthogonal

P @ P == P?  max diff: 1.1102230246251565e-16
residual . u = -2.220446049250313e-16


In [10]:
v = np.array([1.0, 2.0])
pv = P @ v
plt.figure(figsize=(6, 6))
line = np.outer(np.linspace(-0.6, 1.4, 2), u)
plt.plot(line[:, 0], line[:, 1], '--', color='gray', alpha=0.6,
         label='the line of u')
for vec, c, lbl in [(v, 'tab:blue', 'v'), (pv, 'tab:green', 'Pv')]:
    plt.quiver(0, 0, vec[0], vec[1], angles='xy',
               scale_units='xy', scale=1, color=c, label=lbl)
plt.plot([v[0], pv[0]], [v[1], pv[1]], ':', color='tab:red',
         label='residual v - Pv')
plt.gca().set_aspect('equal'); plt.grid(True, alpha=0.3)
plt.legend(); plt.title('Projection: the shadow and the residual')
plt.xlim(-0.5, 3); plt.ylim(-0.5, 3)
plt.savefig('figures/fig_projection.png', dpi=150,
            bbox_inches='tight')
plt.show()

## 2.4 — Systems: the invertible and the cyclic

In [11]:
C = np.array([[1, 0, -1], [-1, 1, 0], [0, -1, 1]])  # cyclic
print('C @ (3,3,3):', C @ np.array([3, 3, 3]))
print('rank of A3:', np.linalg.matrix_rank(A3),
      '  rank of C:', np.linalg.matrix_rank(C))

C @ (3,3,3): [0 0 0]
rank of A3: 3   rank of C: 2


## 2.5/2.6 — Standardization as a transformation, on Ames

In [12]:
import pandas as pd

base = '../ch01/data/'
zoning = pd.read_csv(base + 'zoning.csv')
listing = pd.read_csv(base + 'listing.csv')
sale = pd.read_csv(base + 'sale.csv')
housing = pd.merge(zoning, listing, on='Id')
housing = pd.merge(housing, sale, on='Id').set_index('Id')
numeric = housing.select_dtypes(include='number')
X = numeric.drop(columns='SalePrice')
X = X.dropna(axis=1)                # keep complete columns
print('numeric feature matrix:', X.shape)

numeric feature matrix: (1460, 33)


In [13]:
mu = X.mean().to_numpy()
sigma = X.std(ddof=0).to_numpy()
Z = (X.to_numpy(float) - mu) / sigma
print('column means after:', np.abs(Z.mean(axis=0)).max())
print('column stds after :', np.abs(Z.std(axis=0) - 1).max())

column means after: 3.567435540277722e-14
column stds after : 2.220446049250313e-16


In [14]:
cols = ['GrLivArea', 'OverallQual', 'LotArea', 'YearBuilt']
idx = [X.columns.get_loc(c) for c in cols]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].boxplot([X[c] for c in cols], tick_labels=cols)
axes[0].set_title('raw: incomparable units')
axes[0].tick_params(axis='x', rotation=20)
axes[1].boxplot([Z[:, i] for i in idx], tick_labels=cols)
axes[1].set_title('standardized: one shared scale')
axes[1].tick_params(axis='x', rotation=20)
plt.savefig('figures/fig_standardization.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [15]:
y = housing['SalePrice'].to_numpy(float)
Zsub = Z[:, [X.columns.get_loc('GrLivArea'),
             X.columns.get_loc('OverallQual')]]
w, *_ = np.linalg.lstsq(Zsub, y - y.mean(), rcond=None)
print('weights, dollars per standard deviation:')
print(f'  GrLivArea  : {w[0]:>10,.0f}')
print(f'  OverallQual: {w[1]:>10,.0f}')

weights, dollars per standard deviation:
  GrLivArea  :     29,344
  OverallQual:     45,415
